# CASMI26 v1: explained-intensity ranking + formula prior
MIST-CF-lite, CPU-only, offline. Writes `submission.csv`.

In [ ]:
"""Subformula labelling (MIST-CF lite, RDKit-free).
Assign each MS2 peak a subformula of the candidate precursor formula.
RDBE filter, ppm matching, adduct-adjusted masses.
"""
import numpy as np
from itertools import product

# monoisotopic masses
ELEM_MASS = {
    "C": 12.0, "H": 1.00782503223, "N": 14.00307400443, "O": 15.99491461957,
    "P": 30.9737619985, "S": 31.9720711744, "F": 18.99840316273,
    "Cl": 34.968852682, "Br": 78.9183376, "I": 126.9044719,
    "Na": 22.9897692809, "K": 38.9637074864,
}
ELEM_ORDER = ["C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I"]

ADDUCT_DELTA = {
    "[M+H]+": 1.007276, "[M+Na]+": 22.989218, "[M+K]+": 38.963158,
    "[M+NH4]+": 18.033823, "[M-H]-": -1.007276, "[M+Cl]-": 34.968853,
    "[M+CH2O2-H]-": 44.998201, "[M+C2H4O2-H]-": 59.013851,
    "[M]+": 0.0, "[M-H2O+H]+": -17.003348, "[M-2H2O+H]+": -35.013913,
    "[2M+H]+": None, "[2M+Na]+": None, "[2M-H]-": None, "[M+2H]2+": None,
}


def parse_formula(s):
    """'C9H8N2O2' -> dict. Handles two-letter elements."""
    import re
    out = {}
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", s):
        if el not in ELEM_MASS:
            return None
        out[el] = out.get(el, 0) + (int(n) if n else 1)
    return out


def formula_mass(f):
    return sum(ELEM_MASS[e] * n for e, n in f.items())


def rdbe(f):
    """Ring-double-bond equivalents. None if elements unsupported."""
    c = f.get("C", 0); h = f.get("H", 0); n = f.get("N", 0)
    hal = sum(f.get(e, 0) for e in ("F", "Cl", "Br", "I"))
    for e in f:
        if e not in ("C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I"):
            return None
    return c - (h + hal) / 2 + n / 2 + 1


def enumerate_subformulae(prec_f, max_n=200000):
    """All f ⊆ prec_f with RDBE >= 0, as (counts_tuple, mass). Bounded."""
    keys = [e for e in ELEM_ORDER if e in prec_f]
    counts = [prec_f[e] for e in keys]
    # guard combinatorial explosion (e.g. C30H50...): cap by sampling coarse grid
    total = 1
    for c in counts:
        total *= (c + 1)
    subs = []
    if total <= max_n:
        for combo in product(*[range(c + 1) for c in counts]):
            if all(v == 0 for v in combo):
                continue
            f = dict(zip(keys, combo))
            if (rdbe(f) or -1) < 0:
                continue
            subs.append((combo, sum(ELEM_MASS[e] * n for e, n in zip(keys, combo))))
    else:
        # vectorized random sampling for huge combinatorial spaces
        rng = np.random.default_rng(0)
        k = len(keys)
        cm = np.array(counts)
        draws = rng.integers(0, cm + 1, size=(min(max_n * 3, 600000), k))
        draws = np.unique(draws, axis=0)
        nz = draws[np.any(draws > 0, axis=1)][:max_n]
        idx = {e: i for i, e in enumerate(keys)}
        hal_cols = [idx[e] for e in ("F", "Cl", "Br", "I") if e in idx]
        hal = nz[:, hal_cols].sum(axis=1) if hal_cols else 0
        c = nz[:, idx["C"]] if "C" in idx else 0
        h = nz[:, idx["H"]] if "H" in idx else 0
        n = nz[:, idx["N"]] if "N" in idx else 0
        ok = (c - (h + hal) / 2 + n / 2 + 1) >= 0
        sup = ("C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I")
        if any(e not in sup for e in keys):
            ok = ok & False
        nz = nz[ok][:max_n]
        mv = np.array([ELEM_MASS[e] for e in keys])
        masses = nz @ mv
        subs = [(tuple(row), float(m)) for row, m in zip(nz.tolist(), masses.tolist())]
    return keys, subs


_SUB_CACHE = {}


def subformula_masses(prec_formula_str, max_n=200000):
    """Cached (keys, masses array) for a precursor formula."""
    hit = _SUB_CACHE.get(prec_formula_str)
    if hit is not None:
        return hit
    prec_f = parse_formula(prec_formula_str)
    if prec_f is None:
        return None
    keys, subs = enumerate_subformulae(prec_f, max_n)
    arr = np.array([m for _, m in subs], dtype=float)
    _SUB_CACHE[prec_formula_str] = (keys, arr)
    return keys, arr


def label_peaks(mzs, intens, prec_formula_str, adduct, ppm=15.0, top_n=20):
    """Greedy: for each top-N peak (by intensity), nearest subformula mass within ppm.
    Returns list of (mz, intensity, subformula_mass or None, ppm_err or None).
    Assumes fragments carry precursor adduct (MIST-CF assumption).
    """
    cached = subformula_masses(prec_formula_str)
    if cached is None:
        return [(m, i, None, None) for m, i in zip(mzs, intens)]
    d = ADDUCT_DELTA.get(adduct)
    if d is None:
        return [(m, i, None, None) for m, i in zip(mzs, intens)]
    mzs = np.asarray(mzs, dtype=float); intens = np.asarray(intens, dtype=float)
    order = np.argsort(-intens)[:top_n]
    _, sub_masses = cached
    out = []
    for idx in order:
        target = mzs[idx] - d  # adduct-adjusted neutral fragment mass
        if len(sub_masses) == 0:
            out.append((mzs[idx], intens[idx], None, None))
            continue
        j = int(np.argmin(np.abs(sub_masses - target)))
        err_ppm = abs(sub_masses[j] - target) / max(target, 1e-9) * 1e6
        if err_ppm <= ppm:
            out.append((mzs[idx], intens[idx], float(sub_masses[j]), float(err_ppm)))
        else:
            out.append((mzs[idx], intens[idx], None, None))
    return out


def explained_intensity(mzs, intens, prec_formula_str, adduct, ppm=15.0, top_n=20):
    """Fraction of top-N intensity explained by subformulae. Core v1 feature."""
    labelled = label_peaks(mzs, intens, prec_formula_str, adduct, ppm, top_n)
    tot = sum(i for _, i, _, _ in labelled)
    exp = sum(i for _, i, m, _ in labelled if m is not None)
    n_hit = sum(1 for _, _, m, _ in labelled if m is not None)
    return (exp / tot if tot > 0 else 0.0), n_hit


In [ ]:
"""Formula candidates from structures (database-dependent, tractable).
De novo enumeration is deferred: v1 ranks formulae observed among
mass-window candidate structures by aggregated explained intensity.
"""
import numpy as np
from collections import Counter


class FormulaPrior:
    """FastFilter-lite: P(formula) from train frequencies."""

    def __init__(self):
        self.counts = Counter()
        self.total = 0

    def fit(self, formulae):
        self.counts.update(formulae)
        self.total = sum(self.counts.values())

    def score(self, fstr):
        return float(np.log((self.counts.get(fstr, 0) + 1) / (self.total + len(self.counts) + 1)))


In [ ]:
"""v1 production: explained-intensity ranking + formula prior + fusion.
Writes submission.csv (400 x 25). RDKit-free, CPU.
"""
import numpy as np, pandas as pd, os
from bisect import bisect_left, bisect_right

import glob as _glob
_hits = _glob.glob("/kaggle/input/**/test.parquet", recursive=True)
IN = __import__("os").path.dirname(_hits[0]) if _hits else "data"
OUT = "/kaggle/working" if _hits else "."
print("IN=", IN, "OUT=", OUT)


def neutral_mass(prec, adduct):
    if adduct == "[2M+H]+": return (prec - 1.007276) / 2
    if adduct == "[2M+Na]+": return (prec - 22.989218) / 2
    if adduct == "[2M-H]-": return (prec + 1.007276) / 2
    d = ADDUCT_DELTA.get(adduct)
    return prec - d if d is not None else np.nan


def main():
    test = pd.read_parquet(f"{IN}/test.parquet")
    test["neutral"] = [neutral_mass(p, a) for p, a in zip(test["precursor_mz"], test["adduct"])]
    mol_neutral = test.groupby("molecule_id")["neutral"].median()

    train = pd.read_parquet(f"{IN}/train.parquet",
        columns=["normalized_smiles", "molecular_formula", "adduct",
                 "precursor_mz", "ms2_mzs", "ms2_normalized_intensities"])
    train["neutral"] = [neutral_mass(p, a) for p, a in zip(train["precursor_mz"], train["adduct"])]
    train = train[np.isfinite(train["neutral"].values)]
    prior = FormulaPrior()
    prior.fit(train["molecular_formula"].tolist())

    struct = train.groupby("normalized_smiles").agg(
        mass=("neutral", "median"), formula=("molecular_formula", "first")).reset_index()
    struct = struct.sort_values("mass").reset_index(drop=True)
    masses = struct["mass"].values
    smi = struct["normalized_smiles"].values
    fmap = dict(zip(struct["normalized_smiles"], struct["formula"]))
    # pre-warm subformula cache for common formulae in test mass range (245-460 Da)
    priors = {f: prior.score(f) for f in struct["formula"].unique()}

    rows = []
    for mi, (mol, spectra) in enumerate(test.groupby("molecule_id")):
        qmass = float(mol_neutral.loc[mol])
        tol = qmass * 20 / 1e6
        lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
        pcur = 20
        while hi - lo < 200 and pcur < 500:
            pcur *= 2; tol = qmass * pcur / 1e6
            lo = bisect_left(masses, qmass - tol); hi = bisect_right(masses, qmass + tol)
        cands = smi[lo:hi][:2000]
        qspecs = list(zip(spectra["ms2_mzs"], spectra["ms2_normalized_intensities"], spectra["adduct"]))
        scored = []
        for s in cands:
            f = fmap[s]
            subformula_masses(f)  # warm cache
            best = 0.0
            for mz, it, ad in qspecs:
                v = explained_intensity(mz, it, f, ad)[0]
                if v > best: best = v
            scored.append((best + 0.02 * priors.get(f, -10.0), best, s))
        scored.sort(reverse=True)
        top = [s for _, _, s in scored[:25]]
        rows.append((mol, ";".join(top)))
        if (mi + 1) % 50 == 0: print(f"done {mi+1}/400", flush=True)
    pd.DataFrame(rows, columns=["molecule_id", "smiles"]).to_csv(f"{OUT}/submission.csv", index=False)
    print("wrote submission.csv")


main()
